In [ ]:
!pip -q install mediapipe opencv-python-headless scipy imageio imageio-ffmpeg matplotlib trimesh pyrender "PyOpenGL==3.1.0" pyglet
print("install done")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 75.1 MB/s eta 0:00:00
install done


In [ ]:
import json
import math
import os
import shutil
os.environ["PYOPENGL_PLATFORM"] = "egl"

import cv2
import imageio
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp
import pyrender
import trimesh
from google.colab import files
from scipy.spatial import Delaunay
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

WORK_DIR = "/content/mediapipe_privacy_avatar"
EXTRACT_DIR = f"{WORK_DIR}/extract"
MOSAIC_DIR = f"{EXTRACT_DIR}/mosaic_frames"
META_DIR = f"{EXTRACT_DIR}/metadata"
RENDER_DIR = f"{WORK_DIR}/render"
OUT_DIR = f"{WORK_DIR}/output_frames"
for path in [WORK_DIR, EXTRACT_DIR, MOSAIC_DIR, META_DIR, RENDER_DIR, OUT_DIR]:
    os.makedirs(path, exist_ok=True)

CROP_SIZE = 512

MP_TASK_PATH = f"{WORK_DIR}/face_landmarker.task"
if not os.path.exists(MP_TASK_PATH):
    !wget -q --show-progress -O {MP_TASK_PATH} https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task

base_options = python.BaseOptions(model_asset_path=MP_TASK_PATH)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=True,
)
detector = vision.FaceLandmarker.create_from_options(options)

print("detector ready")


/content/mediapipe_ 100%[===================>]   3.58M  2.15MB/s    in 1.7s    
detector ready


In [ ]:
LEFT_EYE_IDX = [33, 133, 159, 145, 153, 154, 155, 157, 158, 160, 161, 246]
RIGHT_EYE_IDX = [263, 362, 386, 374, 380, 381, 382, 384, 385, 387, 388, 466]
MOUTH_IDX = [0, 13, 14, 17, 37, 39, 40, 61, 78, 80, 81, 82, 84, 87, 88, 91, 95, 146, 178, 181, 185, 191, 267, 269, 270, 291, 308, 310, 311, 312, 314, 317, 318, 321, 324, 375, 402, 405, 409, 415]


def look_at(camera_pos, target, up=np.array([0.0, 1.0, 0.0], dtype=np.float32)):
    camera_pos = np.asarray(camera_pos, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    forward = target - camera_pos
    forward = forward / np.linalg.norm(forward)
    right = np.cross(forward, up)
    right = right / np.linalg.norm(right)
    true_up = np.cross(right, forward)

    pose = np.eye(4, dtype=np.float32)
    pose[:3, 0] = right
    pose[:3, 1] = true_up
    pose[:3, 2] = -forward
    pose[:3, 3] = camera_pos
    return pose


def render_avatar_from_glb(glb_path, image_size=768):
    tri_scene = trimesh.load(glb_path, force="scene")
    if not isinstance(tri_scene, trimesh.Scene):
        tri_scene = trimesh.Scene(tri_scene)

    scene = pyrender.Scene(bg_color=np.array([255, 255, 255, 0], dtype=np.uint8), ambient_light=np.array([0.45, 0.45, 0.45]))

    for node_name in tri_scene.graph.nodes_geometry:
        transform, geom_name = tri_scene.graph[node_name]
        geom = tri_scene.geometry[geom_name]
        mesh = pyrender.Mesh.from_trimesh(geom, smooth=False)
        scene.add(mesh, pose=transform)

    bounds = tri_scene.bounds
    center = bounds.mean(axis=0)
    extents = bounds[1] - bounds[0]
    scale = float(np.max(extents))

    camera_pos = center + np.array([0.0, extents[1] * 0.08, scale * 1.8], dtype=np.float32)
    camera_pose = look_at(camera_pos, center + np.array([0.0, extents[1] * 0.10, 0.0], dtype=np.float32))

    camera = pyrender.PerspectiveCamera(yfov=np.pi / 4.2)
    scene.add(camera, pose=camera_pose)

    for offset in [
        np.array([0.0, scale * 0.3, scale * 1.1], dtype=np.float32),
        np.array([scale * 0.8, scale * 0.1, scale * 0.7], dtype=np.float32),
        np.array([-scale * 0.8, scale * 0.1, scale * 0.7], dtype=np.float32),
    ]:
        light = pyrender.DirectionalLight(color=np.ones(3), intensity=3.0)
        scene.add(light, pose=look_at(center + offset, center))

    renderer = pyrender.OffscreenRenderer(viewport_width=image_size, viewport_height=image_size)
    color, _ = renderer.render(scene)
    renderer.delete()
    return cv2.cvtColor(color, cv2.COLOR_RGB2BGR)


def detect_face(image_bgr):
    h, w = image_bgr.shape[:2]
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = detector.detect(mp_image)
    if not result.face_landmarks:
        return None

    face_landmarks = result.face_landmarks[0]
    points = np.array([[p.x * w, p.y * h] for p in face_landmarks], dtype=np.float32)
    hull = cv2.convexHull(points.astype(np.float32)).reshape(-1, 2)

    blendshape_dict = {}
    if result.face_blendshapes:
        for item in result.face_blendshapes[0]:
            blendshape_dict[item.category_name] = float(item.score)

    matrix = None
    if result.facial_transformation_matrixes:
        matrix = np.array(result.facial_transformation_matrixes[0], dtype=np.float32)

    return {
        "points": points,
        "hull": hull,
        "blendshapes": blendshape_dict,
        "matrix": matrix,
    }


def compute_face_bbox(points, image_shape, pad_ratio=0.22):
    h, w = image_shape[:2]
    x1 = int(np.min(points[:, 0]))
    y1 = int(np.min(points[:, 1]))
    x2 = int(np.max(points[:, 0]))
    y2 = int(np.max(points[:, 1]))
    bw = x2 - x1
    bh = y2 - y1
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    size = max(bw, bh) * (1.0 + pad_ratio)
    half = size / 2.0

    x1 = max(0, int(cx - half))
    y1 = max(0, int(cy - half))
    x2 = min(w, int(cx + half))
    y2 = min(h, int(cy + half))
    return x1, y1, x2, y2


def crop_face(image_bgr, bbox, crop_size=CROP_SIZE):
    x1, y1, x2, y2 = bbox
    crop = image_bgr[y1:y2, x1:x2]
    return cv2.resize(crop, (crop_size, crop_size))


def normalize_points_to_crop(points, bbox, crop_size=CROP_SIZE):
    x1, y1, x2, y2 = bbox
    sx = crop_size / max(1, x2 - x1)
    sy = crop_size / max(1, y2 - y1)
    out = points.copy()
    out[:, 0] = (out[:, 0] - x1) * sx
    out[:, 1] = (out[:, 1] - y1) * sy
    return out


def adjust_points_with_blendshape(points, blendshapes):
    adjusted = points.copy()
    jaw_open = blendshapes.get("jawOpen", 0.0)
    mouth_pucker = blendshapes.get("mouthPucker", 0.0)
    eye_blink_left = blendshapes.get("eyeBlinkLeft", 0.0)
    eye_blink_right = blendshapes.get("eyeBlinkRight", 0.0)

    if jaw_open > 0:
        center = np.mean(adjusted[MOUTH_IDX], axis=0)
        adjusted[MOUTH_IDX, 1] += jaw_open * np.maximum(0, adjusted[MOUTH_IDX, 1] - center[1]) * 0.35

    if mouth_pucker > 0:
        center = np.mean(adjusted[MOUTH_IDX], axis=0)
        adjusted[MOUTH_IDX] = center + (adjusted[MOUTH_IDX] - center) * (1.0 - 0.12 * mouth_pucker)

    if eye_blink_left > 0:
        center = np.mean(adjusted[LEFT_EYE_IDX], axis=0)
        adjusted[LEFT_EYE_IDX, 1] = center[1] + (adjusted[LEFT_EYE_IDX, 1] - center[1]) * (1.0 - 0.85 * eye_blink_left)

    if eye_blink_right > 0:
        center = np.mean(adjusted[RIGHT_EYE_IDX], axis=0)
        adjusted[RIGHT_EYE_IDX, 1] = center[1] + (adjusted[RIGHT_EYE_IDX, 1] - center[1]) * (1.0 - 0.85 * eye_blink_right)

    return adjusted


def apply_affine(src, src_tri, dst_tri, output, mask):
    src_rect = cv2.boundingRect(np.float32([src_tri]))
    dst_rect = cv2.boundingRect(np.float32([dst_tri]))
    x1, y1, w1, h1 = src_rect
    x2, y2, w2, h2 = dst_rect
    if min(w1, h1, w2, h2) <= 0:
        return

    src_crop = src[y1:y1 + h1, x1:x1 + w1]
    src_local = np.float32([[p[0] - x1, p[1] - y1] for p in src_tri])
    dst_local = np.float32([[p[0] - x2, p[1] - y2] for p in dst_tri])

    warp_mat = cv2.getAffineTransform(src_local, dst_local)
    warped = cv2.warpAffine(
        src_crop,
        warp_mat,
        (w2, h2),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT_101,
    )

    tri_mask = np.zeros((h2, w2, 3), dtype=np.float32)
    cv2.fillConvexPoly(tri_mask, np.int32(dst_local), (1.0, 1.0, 1.0), 16, 0)
    output[y2:y2 + h2, x2:x2 + w2] = output[y2:y2 + h2, x2:x2 + w2] * (1.0 - tri_mask) + warped * tri_mask
    mask[y2:y2 + h2, x2:x2 + w2] = np.maximum(mask[y2:y2 + h2, x2:x2 + w2], tri_mask)


def render_avatar_face(source_crop, source_points, driving_points):
    src = np.clip(source_points, 0, source_crop.shape[0] - 1)
    dst = np.clip(driving_points, 0, source_crop.shape[0] - 1)
    tri = Delaunay(dst)
    output = np.zeros_like(source_crop, dtype=np.float32)
    mask = np.zeros_like(source_crop, dtype=np.float32)
    for simplex in tri.simplices:
        apply_affine(source_crop, src[simplex], dst[simplex], output, mask)
    base = source_crop.astype(np.float32)
    blended = output + base * (1.0 - mask)
    return np.clip(blended, 0, 255).astype(np.uint8)


def make_mosaic_frame(frame_bgr, points, blur_ksize=71):
    result = frame_bgr.copy()
    mask = np.zeros(frame_bgr.shape[:2], dtype=np.uint8)
    hull = cv2.convexHull(points.astype(np.int32))
    cv2.fillConvexPoly(mask, hull, 255)
    blurred = cv2.GaussianBlur(frame_bgr, (blur_ksize, blur_ksize), 0)
    result[mask > 0] = blurred[mask > 0]
    return result


def composite_face_on_frame(frame_bgr, face_crop_bgr, bbox, crop_points):
    x1, y1, x2, y2 = bbox
    w = max(1, x2 - x1)
    h = max(1, y2 - y1)

    resized_face = cv2.resize(face_crop_bgr, (w, h))
    mask = np.zeros((CROP_SIZE, CROP_SIZE), dtype=np.uint8)
    hull = cv2.convexHull(crop_points.astype(np.int32))
    cv2.fillConvexPoly(mask, hull, 255)
    mask = cv2.GaussianBlur(mask, (31, 31), 0)
    mask = cv2.resize(mask, (w, h))

    roi = frame_bgr[y1:y2, x1:x2].astype(np.float32)
    alpha = (mask.astype(np.float32) / 255.0)[..., None]
    blended = roi * (1.0 - alpha) + resized_face.astype(np.float32) * alpha
    frame_bgr[y1:y2, x1:x2] = np.clip(blended, 0, 255).astype(np.uint8)
    return frame_bgr


print("helpers ready")


helpers ready


In [ ]:
print("Upload avatar GLB")
avatar_upload = files.upload()
AVATAR_GLB_PATH = next(iter(avatar_upload.keys()))

print("Upload driving video")
driving_upload = files.upload()
DRIVING_VIDEO_PATH = next(iter(driving_upload.keys()))

avatar_render_bgr = render_avatar_from_glb(AVATAR_GLB_PATH, image_size=768)
avatar_render_path = f"{RENDER_DIR}/avatar_render.png"
cv2.imwrite(avatar_render_path, avatar_render_bgr)

avatar_face = detect_face(avatar_render_bgr)
assert avatar_face is not None, "Face not detected on rendered avatar. Try a different GLB or render a frontal avatar image manually."

avatar_bbox = compute_face_bbox(avatar_face["points"], avatar_render_bgr.shape)
avatar_crop = crop_face(avatar_render_bgr, avatar_bbox, CROP_SIZE)
avatar_points = normalize_points_to_crop(avatar_face["points"], avatar_bbox, CROP_SIZE)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(avatar_render_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Avatar render from GLB")
plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(avatar_crop, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Avatar face crop")
plt.tight_layout()
plt.show()

avatar_meta = {
    "avatar_render_path": avatar_render_path,
    "avatar_bbox": list(map(int, avatar_bbox)),
    "avatar_points": avatar_points.tolist(),
}
with open(f"{META_DIR}/avatar_source.json", "w", encoding="utf-8") as f:
    json.dump(avatar_meta, f, ensure_ascii=False, indent=2)

cap = cv2.VideoCapture(DRIVING_VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print("driving frames:", frame_count, "fps:", fps)


Upload avatar GLB


Saving raccoon_head.glb to raccoon_head.glb
Upload driving video


Saving d0.mp4 to d0 (1).mp4


ArgumentError: ("argument 2: TypeError: No array-type handler for type _ctypes.type (value: <cparam 'P' (0x7cbd08417820)>) registered", (1, <cparam 'P' (0x7cbd08417820)>))

In [ ]:
extraction_records = []
cap = cv2.VideoCapture(DRIVING_VIDEO_PATH)
frame_index = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    detected = detect_face(frame)
    mosaic_frame = frame.copy()
    record = {
        "frame_index": frame_index,
        "mosaic_path": f"{MOSAIC_DIR}/frame_{frame_index:05d}.png",
        "has_face": False,
    }

    if detected is not None:
        bbox = compute_face_bbox(detected["points"], frame.shape)
        crop_points = normalize_points_to_crop(detected["points"], bbox, CROP_SIZE)
        crop_points = adjust_points_with_blendshape(crop_points, detected["blendshapes"])
        mosaic_frame = make_mosaic_frame(frame, detected["points"])

        record.update(
            {
                "has_face": True,
                "bbox": list(map(int, bbox)),
                "crop_points": crop_points.tolist(),
                "blendshapes": detected["blendshapes"],
                "matrix": detected["matrix"].tolist() if detected["matrix"] is not None else None,
            }
        )

    cv2.imwrite(record["mosaic_path"], mosaic_frame)
    extraction_records.append(record)
    frame_index += 1

    if frame_index % 25 == 0:
        print("extracted", frame_index)

cap.release()

with open(f"{META_DIR}/driving_mesh_data.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "fps": fps,
            "crop_size": CROP_SIZE,
            "records": extraction_records,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print("saved mesh metadata:", f"{META_DIR}/driving_mesh_data.json")
print("saved mosaic frames:", len(extraction_records))


In [ ]:
metadata_path = f"{META_DIR}/driving_mesh_data.json"
with open(metadata_path, "r", encoding="utf-8") as f:
    driving_data = json.load(f)

with open(f"{META_DIR}/avatar_source.json", "r", encoding="utf-8") as f:
    avatar_meta = json.load(f)

avatar_render_bgr = cv2.imread(avatar_meta["avatar_render_path"])
avatar_bbox = tuple(avatar_meta["avatar_bbox"])
avatar_crop = crop_face(avatar_render_bgr, avatar_bbox, CROP_SIZE)
avatar_points = np.array(avatar_meta["avatar_points"], dtype=np.float32)

output_paths = []
for record in driving_data["records"]:
    mosaic_frame = cv2.imread(record["mosaic_path"])

    if record["has_face"]:
        driving_points = np.array(record["crop_points"], dtype=np.float32)
        bbox = tuple(record["bbox"])
        avatar_face_frame = render_avatar_face(avatar_crop, avatar_points, driving_points)
        mosaic_frame = composite_face_on_frame(mosaic_frame, avatar_face_frame, bbox, driving_points)

    out_path = f"{OUT_DIR}/frame_{record['frame_index']:05d}.png"
    cv2.imwrite(out_path, mosaic_frame)
    output_paths.append(out_path)

    if (record["frame_index"] + 1) % 25 == 0:
        print("composited", record["frame_index"] + 1)

OUTPUT_VIDEO = f"{WORK_DIR}/avatar_privacy_face_swap.mp4"
writer = imageio.get_writer(OUTPUT_VIDEO, fps=driving_data["fps"])
for path in output_paths:
    frame = cv2.imread(path)
    writer.append_data(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
writer.close()

print("video saved:", OUTPUT_VIDEO)
files.download(OUTPUT_VIDEO)


In [ ]:
sample_ids = np.linspace(0, len(output_paths) - 1, num=min(5, len(output_paths)), dtype=int)
plt.figure(figsize=(15, 6))
for i, idx in enumerate(sample_ids, 1):
    frame = cv2.imread(output_paths[idx])
    plt.subplot(1, len(sample_ids), i)
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(idx)
plt.tight_layout()
plt.show()
